# Lab 7: Sequence data mechanics

**DSAN-6600 Deep Learning · Fall 2026**

**Due:** Wednesday Oct 21, 11:59 PM ET · **Target time:** about 75 minutes

---

## What this lab is for

Thursday's lecture said that tokenization, embeddings, padding and masking are
the parts of sequence modelling that do not expire when recurrent networks do.
This lab is those parts, on one small corpus, in four steps:

1. **Tokens and the vocabulary.** Measure how fast a word vocabulary grows,
   what capping it costs you, and what subword tokenization buys instead.
2. **Padding.** Build a rectangular batch out of ragged sentences and measure
   how much of it is nothing.
3. **Masking.** Pool that batch with and without a mask, predict the error
   before you measure it, and then show that an unmasked loss prefers a model
   that gets every real word wrong.
4. **Building the batch.** Write the `collate_fn` that a `DataLoader` needs for
   ragged data, and measure what sorting by length saves.

Nothing is trained. There is no LSTM in this lab, on purpose. Next week's
transformer needs every one of these four things and none of the recurrence.

## How this lab is graded

- **The notebook is graded for completion, not for correctness.** Submit it,
  with all cells run and outputs visible, and you get the points.
- **Every number in this lab is reproducible to the digit.** There is no
  training and no GPU, so if your output differs from a classmate's, one of
  you has a bug. That is different from Lab 6 and it is a feature.
- **The understanding is assessed on Quiz 8**, the Thursday after the due date.
- **Solutions are posted after the deadline.**

## Using AI on this lab

Allowed and expected, with disclosure; there is a cell for that at the end.
An agent will write you a correct `collate_fn` in one shot. It is much less
likely to tell you why the masked and unmasked pooled vectors differ by
exactly the fraction they do, and Part 3c asks you for that number in advance.

## Step 0: Setup

::: {.callout-note}
## No GPU this week
Nothing here is trained, so a CPU runtime is fine and the whole notebook runs
in well under a minute. If you still have a GPU runtime on from Lab 6, it does
no harm.
:::

**Download.** One 1.1 MB text file. If your network blocks raw.githubusercontent.com,
the cell falls back to a mirror.

In [ ]:
import re
import urllib.request
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 6600
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

PAD, UNK = 0, 1          # reserved ids, fixed for the whole lab

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print(f"torch {torch.__version__} on cpu")

In [ ]:
URLS = [
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/"
    "tinyshakespeare/input.txt",
    "https://ocw.mit.edu/ans7870/6/6.006/s08/lecturenotes/files/t8.shakespeare.txt",
]

raw = None
for u in URLS:
    try:
        raw = urllib.request.urlopen(u, timeout=30).read().decode("utf-8")
        print(f"got {len(raw):,} characters from {u.split('/')[2]}")
        break
    except Exception as e:
        print(f"  {u.split('/')[2]} failed: {e}")
assert raw, "could not download the corpus; check your network"

---

# Part 1 · Tokens and the vocabulary

## 1a · Getting sentences out of the file

The file is a play, so most lines are either a speaker label (`First Citizen:`)
or a fragment of one speech. Neither is a sentence, and lines in a play are all
roughly the same length, which would hide the thing this lab is about. So:
drop the speaker labels, join everything back together, and split on sentence
punctuation.

In [ ]:
lines = [l.strip() for l in raw.split("\n") if l.strip()]
dialogue = [l for l in lines if not re.fullmatch(r"[A-Za-z' ]+:", l)]
sents = [s.strip() for s in re.split(r"(?<=[.!?;])\s+", " ".join(dialogue))
         if len(s.split()) >= 2]

wl = np.array([len(s.split()) for s in sents])
print(f"{len(lines):,} lines -> {len(sents):,} sentences "
      f"({len(lines) - len(dialogue):,} speaker labels dropped)")
print(f"words per sentence: min {wl.min()}  median {int(np.median(wl))}  "
      f"mean {wl.mean():.1f}  p90 {int(np.percentile(wl, 90))}  max {wl.max()}")
for s in sents[:3]:
    print(f"   {s[:76]!r}")

That length distribution is the lab. The median sentence is 9 words and the
longest is 192, so any batch you build is mostly going to be empty space.

## 1b · One sentence, three granularities

Tokenization splits text into pieces. The pieces can be characters, words, or
subwords, and the choice trades sequence length against vocabulary size.

In [ ]:
S = sents[2]
print(f"{S!r}\n")

# TODO: count the tokens this sentence becomes at each granularity.
#   Characters: every character counts, including spaces. len() does it.
#   Words: split on whitespace.

n_chars = None
n_words = None

print(f"  as characters : {n_chars} tokens")
print(f"  as words      : {n_words} tokens")

Characters give five times as many tokens, and work grows with sequence
length. What they buy is a vocabulary of about 65 symbols that can spell
anything. Words go the other way, and the next cell shows how far.

## 1c · A word vocabulary does not stop growing

Take the corpus as a stream of lowercase words and count how many *distinct*
words you have seen after the first 5%, 10%, 25%, 50% and 100% of it. If a
word vocabulary were a fixed-size thing, this curve would flatten.

In [ ]:
words = [w for s in sents for w in s.lower().split()]
print(f"{len(words):,} word tokens\n")

# TODO: for each fraction, count the distinct words in the first that-many
#   tokens of the stream. set() of a slice, then len().

for frac in [0.05, 0.10, 0.25, 0.50, 1.00]:
    n = int(len(words) * frac)
    distinct = None
    print(f"  first {frac:>5.0%} ({n:>7,} tokens): {distinct:>6,} distinct words")

Doubling the corpus from 50% to 100% added 8,269 new words. It is still
climbing when the corpus runs out, and it would keep climbing on a bigger one,
because names, typos, numbers and inflections never run out.

## 1d · What a vocabulary cap costs

So you cap it: keep the `k` most frequent words and map everything else to
`<unk>`. Measure what that throws away.

In [ ]:
counts = Counter(words)
print(f"full vocabulary: {len(counts):,} distinct words\n")

# TODO: for each cap, work out the share of word TOKENS (not types) that fall
#   outside the top-k and become <unk>.
#   counts.most_common(cap) gives the kept words; sum the counts of the rest
#   and divide by len(words).

for cap in [1_000, 2_000, 5_000, 10_000, 20_000]:
    oov_rate = None
    print(f"  cap {cap:>6,}: {oov_rate:>7.2%} of tokens become <unk>")

A 10,000 word vocabulary, which is not small, still destroys 6.92% of the
corpus. Every one of those tokens becomes the same symbol, so the model cannot
tell `Coriolanus` from `apothecary` from a typo.

## 1e · Subwords: a fixed vocabulary that spells everything

Byte pair encoding starts from characters and repeatedly merges the most
frequent adjacent pair, stopping when it hits the vocabulary size you asked
for. Frequent words survive whole; rare ones get split into pieces that are
themselves in the vocabulary. Nothing is unspellable, so nothing becomes
`<unk>`.

This trains a real BPE tokenizer on the corpus, three times, at three
vocabulary sizes.

In [ ]:
from tokenizers import Tokenizer, models, pre_tokenizers, trainers

for cap in [500, 1_000, 4_000]:
    tok = Tokenizer(models.BPE(unk_token="<unk>"))
    tok.pre_tokenizer = pre_tokenizers.Whitespace()
    tok.train_from_iterator(sents, trainers.BpeTrainer(
        vocab_size=cap, special_tokens=["<pad>", "<unk>"], show_progress=False))

    pieces = tok.encode(S).tokens
    sample = sents[:2000]
    n_unk = sum(t.tokens.count("<unk>") for t in map(tok.encode, sample))
    n_tok = sum(len(t.tokens) for t in map(tok.encode, sample))
    print(f"  vocab {cap:>5,}: {len(pieces):>3} subwords for the test sentence, "
          f"{n_unk / n_tok:.4%} <unk> over 2,000 sentences")
    if cap == 1_000:
        print(f"           {pieces}")

Compare that with 1d. A word vocabulary of 10,000 drops 6.92% of tokens. BPE
with a vocabulary of **500** drops none, because `resolved` can always fall
back to `res` + `ol` + `ved`. That is why every model you will use from here
on is subword.

## 1f · The vocabulary this lab will use

Parts 2 through 4 need integer ids, so build the lookup. Word level, capped at
8,000, with the two reserved ids fixed at the top. **Id 0 is `<pad>`**, and
that convention is load-bearing for the rest of the lab.

In [ ]:
vocab = ["<pad>", "<unk>"] + [w for w, _ in counts.most_common(8_000)]
stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for w, i in stoi.items()}
V = len(vocab)


def encode(s):
    """Sentence -> list of ids, unknown words -> UNK."""
    return [stoi.get(w, UNK) for w in s.lower().split()]


print(f"vocabulary: {V:,} ids   PAD={PAD} ({itos[PAD]!r})  UNK={UNK} ({itos[UNK]!r})")
print(f"  {S!r}")
print(f"  -> {encode(S)}")
print(f"  -> {[itos[i] for i in encode(S)]}")

Look at the last token. `famish?` came back as `<unk>`, and the word `famish`
is common enough to be well inside a 8,000 word vocabulary. Splitting on
whitespace leaves the question mark glued to the word, so `famish?` and
`famish` are two different entries and the corpus only ever contains one of
them often enough to keep.

That is what a *pre-tokenizer* is for, and it is why the BPE tokenizer in 1e
was given `pre_tokenizers.Whitespace()`, which separates punctuation first.
This lab keeps the naive version so the failure stays visible.

---

# Part 2 · Padding a ragged batch

A batch has to be a rectangle, because a tensor has a shape. Sentences are not
the same length. Padding appends the reserved id to every sequence until they
all reach the length of the longest one in the batch.

## 2a · Eight sentences

In [ ]:
pick = rng.choice(len(sents), 8, replace=False)
batch = [encode(sents[i]) for i in pick]
lengths = [len(b) for b in batch]

print(f"lengths: {lengths}")
print(f"longest: T = {max(lengths)}")
for i, b in zip(pick, batch):
    print(f"  {len(b):>3} | {sents[i][:64]}")

## 2b · Making it rectangular

In [ ]:
T = max(lengths)

# TODO: build the padded id tensor X, shape (8, T), dtype long.
#   Start from torch.full filled with PAD, then write each sequence into the
#   front of its row. Row r gets batch[r] in columns 0 to len(batch[r]).

X = None

print(f"X {tuple(X.shape)} {X.dtype}")
print(X[:4, :14])

## 2c · How much of that batch is nothing

In [ ]:
# TODO: count the cells, the real tokens and the padding, and report the
#   padding as a share of the whole rectangle.
#   X.numel() is the cell count; (X != PAD).sum() counts the real ones.

n_cells = None
n_real = None
n_pad = None

print(f"  cells   {n_cells:>5,}")
print(f"  real    {n_real:>5,}")
print(f"  padding {n_pad:>5,}   ({n_pad / n_cells:.1%} of the batch)")

More than half the batch means "nothing here". One 24 word sentence sets `T`
for a batch whose median is 9, and everybody else pays for it. Part 4 fixes
most of that with two lines.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 2.6))
ax.imshow((X != PAD).numpy(), aspect="auto", cmap="Blues", vmin=0, vmax=1.4)
ax.set_xticks(range(0, T, 2))
ax.set_yticks(range(len(batch)))
ax.set_yticklabels([f"{l} words" for l in lengths], fontsize=8)
ax.set_xlabel("position in the sequence")
ax.set_title(f"Dark is a real token, white is PAD "
             f"({n_pad}/{n_cells} = {n_pad / n_cells:.0%} padding)")
ax.grid(False)
plt.tight_layout()
plt.show()

---

# Part 3 · Masking, measured

A mask is a boolean tensor the same shape as the batch, `True` where the token
is real. Without one, every layer downstream reads `PAD` as a word.

## 3a · The mask, and the embedding that knows about PAD

In [ ]:
mask = X != PAD
assert int(mask.sum()) == n_real

torch.manual_seed(SEED)
emb = nn.Embedding(V, 64, padding_idx=PAD)
E = emb(X)

print(f"mask {tuple(mask.shape)} {mask.dtype}, {int(mask.sum())} True")
print(f"E    {tuple(E.shape)}")
print(f"\npadding_idx={PAD} pins that row to zero and keeps it there:")
print(f"  emb.weight[PAD].abs().max() = {emb.weight[PAD].abs().max():.4f}")
print(f"  E[0, -1].abs().max()        = {E[0, -1].abs().max():.4f}   "
      f"(row 0 is {lengths[0]} long, so this position is PAD)")

`padding_idx` does two things: it initializes that row to zeros and it stops
gradient from ever reaching it. So `PAD` embeds to the zero vector, now and
after any amount of training.

## 3b · Predict the error before you measure it

You are about to mean-pool this batch two ways: `E.mean(dim=1)`, which divides
by `T`, and the masked version, which divides by the real length.

Because `PAD` embeds to **zero**, the two differ in a way you can work out
exactly, with no code. Call the sum of the real token vectors in a row $v$,
the row's length $\ell$ and the batch width $T$:

$$\text{naive} = \frac{v}{T} \qquad \text{masked} = \frac{v}{\ell}
  \qquad\Rightarrow\qquad \text{naive} = \frac{\ell}{T}\,\text{masked}$$

So the naive pooled vector is the right one, shrunk by $\ell / T$, and the
relative error $\lVert \text{naive} - \text{masked}\rVert /
\lVert \text{masked} \rVert$ is exactly $1 - \ell / T$.

**Write your predictions in the cell below before you run the next one.**

In [ ]:
# TODO: fill in the predicted relative error for the shortest and the longest
#   row, using 1 - length/T. No tensors needed; this is arithmetic.

T_ = max(lengths)
shortest, longest = min(lengths), max(lengths)

pred_shortest = None
pred_longest = None

print(f"  T = {T_}")
print(f"  shortest row ({shortest} words): predict {pred_shortest:.1%} error")
print(f"  longest  row ({longest} words): predict {pred_longest:.1%} error")

## 3c · Now measure it

In [ ]:
naive = E.mean(dim=1)

# TODO: pool with the mask. Zero out the PAD positions, sum over the sequence
#   axis, and divide each row by its own real length.
#   mask.unsqueeze(-1) broadcasts the mask over the 64 embedding features;
#   mask.sum(dim=1, keepdim=True) is the per-row length.

masked = None

err = (naive - masked).norm(dim=1) / masked.norm(dim=1)

print("  len   relative error   1 - len/T")
for l, e in sorted(zip(lengths, err.tolist())):
    print(f"  {l:>4}   {e:>12.1%}   {1 - l / T:>9.1%}")
print(f"\n  correlation between length and error: "
      f"{np.corrcoef(lengths, err.detach().numpy())[0, 1]:+.2f}")

The two columns agree to the digit, and the correlation is exactly $-1$. This
is the tell from the lecture, now with a mechanism: **a masking bug damages
short examples in proportion to how short they are, and leaves the longest one
in each batch untouched.** The longest row has no padding, so it is correct
even with the bug, which is why the batch average looks only mildly wrong.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.2))
o = np.argsort(lengths)
ax.plot(np.array(lengths)[o], 100 * err.detach().numpy()[o],
        "o-", color="#C44E52", label="measured")
ax.plot(np.array(lengths)[o], 100 * (1 - np.array(lengths)[o] / T),
        "--", color="#4C72B0", label=r"$1 - \ell/T$")
ax.set_xlabel("sentence length in tokens")
ax.set_ylabel("relative error (%)")
ax.set_title(f"Unmasked mean-pool error against length (T = {T})")
ax.legend()
plt.tight_layout()
plt.show()

## 3d · The same bug in the loss

Masking the pooling is not enough; the loss needs it too. Rather than train
something, compare two hypothetical models on this batch directly.

- **Model A** predicts every `PAD` position perfectly and gets every real word
  wrong.
- **Model B** predicts every real word perfectly and gets every `PAD` wrong.

Model B is the only one of the two that has learned anything.

In [ ]:
def logits_that_are_right_on(which):
    """Confident logits: right on `pad` positions, or right on `real` ones."""
    lg = torch.zeros(len(batch), T, V)
    want = (X == PAD) if which == "pad" else (X != PAD)
    lg.scatter_(2, X.unsqueeze(-1), torch.where(want, 20.0, 0.0).unsqueeze(-1))
    return lg


# TODO: score both models two ways, with and without ignore_index.
#   F.cross_entropy wants (N*T, V) and (N*T,), so reshape both.
#   The masked version passes ignore_index=PAD.

for name, which in [("A: every PAD right, every word wrong", "pad"),
                    ("B: every word right, every PAD wrong", "real")]:
    lg = logits_that_are_right_on(which)
    unmasked = None
    with_ignore = None
    print(f"  {name:38}  unmasked {unmasked:6.3f}   ignore_index {with_ignore:6.3f}")

Read the unmasked column. **Model A scores the lower loss**, so gradient
descent on an unmasked loss will walk towards the model that gets every real
word wrong. It is not a rounding error either; it follows directly from 52.1%
of the target cells being `PAD`, so more than half the available credit is for
predicting "nothing here".

The `ignore_index` column orders them the way you would want, 8.987 against
0.000.

Three places the mask has to be applied, and forgetting any one of them is
silent:

| where | what it looks like |
|---|---|
| pooling | divide by the real count, not `T` |
| the loss | `ignore_index=PAD` |
| attention, next week | `PAD` positions go to $-\infty$ before the softmax |

---

# Part 4 · Building the batch

A `Dataset` returns one example. The default collation stacks them, which
needs matching shapes, and ragged sequences do not have them.

## 4a · Watch it fail

In [ ]:
class Sentences(Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        return torch.tensor(encode(self.texts[i]))


ds = Sentences(sents)
try:
    next(iter(DataLoader(ds, batch_size=8)))
except RuntimeError as e:
    print(f"RuntimeError: {str(e)[:150]}")

## 4b · The collate_fn

`collate_fn` receives the list of examples and returns whatever the model
wants. For ragged sequences that means pad to the longest member **of this
batch**, and hand back the mask alongside the ids so nothing downstream has to
guess.

In [ ]:
# TODO: pad a list of 1-D id tensors into (ids, mask).
#   T is the longest sequence in THIS batch, not a global maximum.
#   Return ids as long and mask as bool, both (len(items), T).

def collate(items):
    T = None
    ids = None
    for r, s in enumerate(items):
        pass
    mask = None
    return ids, mask


ids, msk = next(iter(DataLoader(ds, batch_size=8, collate_fn=collate)))
print(f"ids  {tuple(ids.shape)} {ids.dtype}")
print(f"mask {tuple(msk.shape)} {msk.dtype}")
print(f"real tokens {int(msk.sum())} of {msk.numel()} cells")

## 4c · Bucketing

`T` is set by the longest member of each batch, so if you sort by length first
the batches become internally uniform and the padding mostly disappears. That
is called bucketing. Measure what it saves.

In [ ]:
sample = [len(encode(sents[i])) for i in rng.permutation(len(sents))[:2048]]


# TODO: given a list of lengths and a batch size, return the share of cells
#   that would be padding. Walk the list in chunks of `bs`; each chunk costs
#   len(chunk) * max(chunk) cells and carries sum(chunk) real tokens.

def padding_share(lens, bs):
    cells = real = 0
    for i in range(0, len(lens), bs):
        chunk = lens[i:i + bs]
    return None


for bs in [8, 32]:
    shuffled = padding_share(sample, bs)
    sorted_ = padding_share(sorted(sample), bs)
    print(f"  batch {bs:>3}: shuffled {shuffled:>6.1%} padding, "
          f"sorted {sorted_:>6.1%}")

At batch 8 that is 60.2% of the work going to padding against 2.2%. You would
not sort the whole dataset in practice, because then every epoch sees the same
batches; the usual compromise is to shuffle, take a pool of a few hundred, sort
inside the pool and batch from that.

::: {.callout-warning}
## Sorting is not the only way order matters
Bucketing reorders sentences, which is harmless because they are independent.
A **time series** is not independent. Split it on time, train on the past and
validate on the future. A random split lets the model see tomorrow while it
learns yesterday, and the validation score it reports is then meaningless.
:::

---

# Part 5 · What this lab showed

Seven questions. If you can answer them, you are ready for Quiz 8.

1. **Why a word-level vocabulary cannot be made large enough**, in terms of the
   curve you measured in 1c.
2. **What `<unk>` costs** at a 10,000 word cap, and why subword tokenization
   gets that to zero with a vocabulary twenty times smaller.
3. **What the pad id is for** and why it is reserved rather than just unused.
4. **Why the unmasked pooled vector is wrong by exactly $1 - \ell/T$**, and
   what that tells you about which examples a masking bug damages.
5. **Why a masking bug is hard to notice** from a validation average.
6. **Why an unmasked loss prefers Model A**, the one that gets every real word
   wrong, and what fraction of the batch has to be padding for that to happen.
7. **What `collate_fn` is for**, and what sorting by length changes about the
   amount of work the GPU does.

Every one of these carries into next week unchanged. The transformer needs
tokens, a vocabulary, embeddings, padding and masks exactly as they are here;
the only thing it discards is the recurrence.

---

# Part 6 · AI disclosure

Required if you used any generative AI on this lab. Name the tool, say where you
used it, and say what for. "None" is a perfectly good answer.

*Tool(s):*

*Where:*

*What for:*

---

# Submitting

1. **Restart the kernel and Run All.** Confirm it runs top to bottom.
2. Check that every plot and every printed block is visible.
3. Render to **HTML or PDF** with resources embedded.
4. Submit the rendered file on Canvas. Due **Wed Oct 21, 11:59 PM ET**.

The notebook is graded for completion, so this is mostly a formality. Part 5 is
your study guide for the quiz that assesses the understanding.

::: {.callout-note}
## Your numbers should match exactly
No training and no GPU means no nondeterminism. Every figure in this notebook
is reproducible to the digit, so a number that differs from a classmate's is a
bug worth finding.
:::